# M2: asymmetric rotor, operator library and fit

A thin driver. It covers:

1. The ModelSpec, its canonical hash and the operator cache.
2. Watson A-reduced rotor blocks with their compile certificates.
3. Synthetic assigned levels (illustrative H2CO-like constants, **not fitted values**).
4. A Levenberg-Marquardt fit with Hellmann-Feynman Jacobians, then parameter recovery.
5. Gauge: a redundant operator shows up as a reported null direction.
6. A result record validated against `schemas/result.schema.json`.

In [ ]:
import json, os
from pathlib import Path
import numpy as np
import riss_drucker as rd
from riss_drucker.rotor import asym_top_labels

WORK = Path(os.environ.get("RISS_DRUCKER_WORK", "/kaggle/working" if Path("/kaggle/working").is_dir() else "."))
spec = rd.ModelSpec.asym_rotor(j_max=20, reduction="A", axis="Ir", order=4)
print("spec hash:", spec.hash())
model, info = rd.compile_cached(spec)
print("cache:", info)
print("parameters:", model.param_names, "| blocks:", model.num_blocks)
for c in model.compile_certificates:
    print(f"  {c['name']:18s} passed={c['passed']}  value={c['value']:.2e}  bound={c['bound']:.1e}")

In [ ]:
TRUE = {"A": 9.405, "B": 1.2953, "C": 1.1342, "DeltaJ": 2.4e-6, "DeltaJK": 9.7e-5,
        "DeltaK": 4.4e-3, "deltaJ": 3.4e-7, "deltaK": 2.7e-5}   # illustrative, cm^-1
labels = asym_top_labels(model)
obs = rd.synth.synthesize_levels(model, TRUE, noise=1e-4, seed=0, labels=labels)
print(len(obs), "synthetic levels, e.g.", [(o.label, round(o.energy, 5)) for o in obs[:4]])
start = rd.synth.perturb_theta(TRUE, rel=0.02, seed=3)
res = rd.fit(model, obs, start)
print(f"converged={res.converged} certified={res.certified} iterations={res.iterations} "
      f"chi2/dof={res.reduced_chi2:.3f} rms={res.rms:.2e} cm-1 rank={res.rank}")
print(f"{'param':8s} {'true':>14s} {'fit':>14s} {'sigma':>10s} {'pull':>6s}")
for n in model.param_names:
    print(f"{n:8s} {TRUE[n]:14.8g} {res.theta[n]:14.8g} {res.sigma[n]:10.2e} {(res.theta[n]-TRUE[n])/res.sigma[n]:6.2f}")

In [ ]:
try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
except ImportError:
    plt = None
if plt is not None:
    J = np.array([int(l["label"].split("(")[0]) for l in res.levels])
    Ka = np.array([int(l["label"].split("(")[1].split(",")[0]) for l in res.levels])
    resid = np.array([l["residual"] for l in res.levels]) * 1e4
    fig, ax = plt.subplots(figsize=(7, 3.8))
    sc = ax.scatter(J + 0.15 * (Ka % 3 - 1), resid, c=Ka, s=10, cmap="viridis")
    ax.axhline(0, color="k", lw=0.6)
    ax.set(xlabel="J", ylabel="obs - calc / 10$^{-4}$ cm$^{-1}$", title="Residuals vs J (colour: Ka)")
    fig.colorbar(sc, label="Ka")
    fig.tight_layout()
    fig.savefig(WORK / "02_residuals_vs_J.png", dpi=150)
    print("saved", WORK / "02_residuals_vs_J.png")

## Gauge: a redundant operator gives a null direction

J² = J_a² + J_b² + J_c², so adding a `J2` operator makes θ + t·(1, 1, 1, −1)/2 leave every level unchanged. The fit reports that direction and fixes the gauge at the starting point. Steps and the covariance are restricted to the identifiable subspace. The parameters along the null direction get no σ.

In [ ]:
mg = rd.compile_asym_rotor(j_max=10, order=4, extra_operators=["J2"])
thg = {**TRUE, "J2": 0.0}
obs_g = rd.synth.synthesize_levels(mg, thg, noise=1e-4, seed=1)
rg = rd.fit(mg, obs_g, {**thg, "A": 9.3, "J2": 0.05})
print("rank", rg.rank, "of", len(mg.param_names), "| null directions:", rg.null_directions)
print("sigma:", {k: rg.sigma[k] for k in ("A", "B", "C", "J2")})
print("invariant A + J2 =", rg.theta["A"] + rg.theta["J2"], "(true A =", TRUE["A"], ")")

In [ ]:
final = rd.solve(model, res.theta, nlowest=21)
rec = rd.fit_result_record(res, spec, model=model, final_solution=final,
                           run_record=rd.run_record(spec_hash=spec.hash(), seeds={"synth": 0, "perturb": 3}))
try:
    rd.validate_result(rec)
    print("result record validates against schemas/result.schema.json")
except ImportError:
    print("jsonschema not installed; skipped validation")
out = WORK / "result_m2.json"
out.write_text(json.dumps(rec, indent=1))
print("wrote", out, f"({out.stat().st_size // 1024} kB)")